In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

torch.cuda.empty_cache()

# model_name = "meta-llama/Llama-3.2-1B"
model_name = "Qwen/Qwen3-0.6B"
model_save_dir = "model/downloaded"
peft_config = LoraConfig(
    r = 8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none" ,
    target_modules=["q_proj","v_proj", "o_proj", "k_proj", "up_proj", "down_proj", "gate_proj"],
)

os.makedirs(model_save_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="cuda",
    cache_dir=model_save_dir
)
peft_model = get_peft_model(model, peft_config)

In [4]:
from pruning.utils import find_layers, get_layers
from peft.tuners.lora import Linear

def get_comp_norm(comp, dim):
    comp_weight = comp.base_layer.weight + torch.matmul(comp.lora_B.default.weight,
                                                        comp.lora_A.default.weight)
    # comp_weight has shape [8192, 2048]. Compute L2 norm over dim=0, yielding (1,2048),
    # then transpose to get (2048,1)
    # return torch.norm(comp_weight, p=2, dim=0, keepdim=True).transpose(0, 1).flip(dims=[0])
    # return torch.norm(comp_weight, p=2, dim=0, keepdim=True).transpose(0, 1)
    return torch.norm(comp_weight, p=2, dim=dim).reshape(1, -1)

layers = get_layers(peft_model)
subset = find_layers(layers[0], layers=(Linear,))
num_q_heads = model.config.num_attention_heads
head_dim = model.config.head_dim
num_kv_groups = model.config.num_key_value_heads
effective_weight = subset['self_attn.o_proj'].base_layer.weight
X_hat = torch.rand([1,2048]).to(effective_weight.device)
norm_gate = get_comp_norm(comp = subset["mlp.gate_proj"], dim=0)
norm_up   = get_comp_norm(comp = subset["mlp.up_proj"], dim=0)
norm_gate = norm_gate.reshape(-1,1)
norm_up   = norm_up.reshape(-1,1)
norm = (norm_gate + norm_up)/2
print(f"effective_weight shape:    {effective_weight.shape}")
print(f"X_hat shape:               {X_hat.shape}")
print(f"norm_gate shape:           {norm_gate.shape}")
print(f"norm_up shape:             {norm_up.shape}")
importance = torch.abs(effective_weight) * X_hat * norm

effective_weight shape:    torch.Size([1024, 2048])
X_hat shape:               torch.Size([1, 2048])
norm_gate shape:           torch.Size([1024, 1])
norm_up shape:             torch.Size([1024, 1])


In [ ]:
from pruning.utils import find_layers, get_layers
from peft.tuners.lora import Linear

layers = get_layers(peft_model)
subsets = find_layers(layers[0], layers=(Linear,))

In [9]:
num_q_heads = model.config.num_attention_heads
head_dim = model.config.head_dim
num_kv_groups = model.config.num_key_value_heads
effective_weight = subsets['self_attn.v_proj'].base_layer.weight
X_hat = torch.rand([1,1024]).to(effective_weight.device)
comp_weight = subsets['self_attn.o_proj'].base_layer.weight
out_features, in_features = effective_weight.shape
g = num_q_heads // num_kv_groups
Wv_h = effective_weight.view(num_kv_groups, head_dim, in_features)
Wo_blocks = comp_weight.view(comp_weight.shape[0], num_q_heads, head_dim)
Wo_grouped = Wo_blocks.view(comp_weight.shape[0], num_kv_groups, g, head_dim)
tmp = torch.norm(Wo_grouped, p=2, dim=0)          # norm over out_dim -> [h_kv, g, d_h]
comp_kv = torch.sqrt(tmp.pow(2).mean(dim=1))       # RMS over g -> [h_kv, d_h]
v_proj_head_rows = effective_weight.shape[0] // num_kv_groups
comp_head_rows = comp_weight.shape[0] // num_kv_groups
o_proj_grouped = comp_weight.view(num_kv_groups, comp_head_rows, comp_weight.shape[-1])
comp_norm = torch.norm(o_proj_grouped, p=2, dim=1, keepdim=True)  # shape: (8, 1, 2048)
importance_grouped = torch.abs(Wv_h) * comp_kv.unsqueeze(-1) * X_hat
importance = importance_grouped.view(out_features, in_features)

In [10]:
importance.shape

torch.Size([1024, 1024])

In [10]:
from evaluation.perplexity import eval_ppl

ppl = eval_ppl(model = model, tokenizer=tokenizer, dataset="wikitext2", seqlen=2048)
print (f"Perplexity before pruning: {ppl}")

evaluating on wikitext2


Token indices sequence length is longer than the specified maximum sequence length for this model (2541000 > 131072). Running this sequence through the model will result in indexing errors


block 0/147
block 50/147
block 100/147
Perplexity before pruning: 19.897984121432994


In [4]:
from data.datasets import get_loaders
from pruning.utils import prepare_calibration_input

dataloader, _ = get_loaders(
    name="wikitext2",
    nsamples=32,
    seqlen=2048,
    tokenizer=tokenizer)
inps, outs, attention_mask, position_ids = prepare_calibration_input(peft_model, dataloader)

In [13]:
from pruning.prune import prune_wanda
from pruning.utils import check_sparsity

prune_wanda(model =  peft_model,
            tokenizer = tokenizer,
            sparsity_ratio = 0.3,
            nsamples= 128,
            seqlen= 2048,
            seed = 42)

check_sparsity(model)


loading calibration data


dataset loading complete
pruning layer 0 name self_attn.q_proj.base_layer
pruning layer 0 name self_attn.q_proj.lora_A.default
pruning layer 0 name self_attn.q_proj.lora_B.default
pruning layer 0 name self_attn.k_proj.base_layer
pruning layer 0 name self_attn.k_proj.lora_A.default
pruning layer 0 name self_attn.k_proj.lora_B.default
pruning layer 0 name self_attn.v_proj.base_layer
pruning layer 0 name self_attn.v_proj.lora_A.default
pruning layer 0 name self_attn.v_proj.lora_B.default
pruning layer 0 name self_attn.o_proj.base_layer
pruning layer 0 name self_attn.o_proj.lora_A.default
pruning layer 0 name self_attn.o_proj.lora_B.default
pruning layer 0 name mlp.gate_proj.base_layer
pruning layer 0 name mlp.gate_proj.lora_A.default
pruning layer 0 name mlp.gate_proj.lora_B.default
pruning layer 0 name mlp.up_proj.base_layer
pruning layer 0 name mlp.up_proj.lora_A.default
pruning layer 0 name mlp.up_proj.lora_B.default
pruning layer 0 name mlp.down_proj.base_layer
pruning layer 0 name ml

0.30413133206745624

In [4]:
from evaluation.perplexity import eval_ppl

ppl = eval_ppl(model = model, tokenizer=tokenizer, dataset="wikitext2", seqlen=2048)
print (f"Perplexity after pruning: {ppl}")

evaluating on wikitext2


block 0/147
block 50/147
block 100/147
Perplexity after pruning: 21.038264827714453


In [ ]:
from data.datasets import get_loaders
from data.utils import prepare_calibration_input

print("Loading data...")
train_data, val_data = get_loaders(name = 'harrison', nsamples=128, seed=42, seqlen=2048, tokenizer=tokenizer)
print("Preparing calibration input...")
inps, outs, attention_mask, position_ids = prepare_calibration_input(peft_model, train_data, device='cuda', max_samples=32)

Loading data...
Preparing calibration input...


In [2]:
from pruning.WrappedGPT import WrappedGPT
from pruning.utils import find_layers, get_layers, prepare_calibration_input
import torch.nn as nn
from pruning.utils import layer_forward, merge_lora_into_base, reset_lora
from data.datasets import get_loaders
from peft.tuners.lora import Linear
def foresight_prune(model, 
                prune_ratio,
                mask_lr,
                nsamples=128,
                seed=42,
                seqlen=2048,
                tokenizer=None,
                device=None,
                dataset_name = 'c4'):
    use_cache = model.config.use_cache
    model.config.use_cache = False

    num_q_heads = model.config.num_attention_heads
    head_dim = model.config.head_dim
    num_kv_groups = model.config.num_key_value_heads

    model.eval()
    if device is None:
        device = next(model.parameters()).device

    print("loading data")
    dataloader, _ = get_loaders(
        dataset_name,
        nsamples=nsamples,
        seed=seed,
        seqlen=seqlen,
        tokenizer=tokenizer,
    )
    print("dataset loading complete")

    print("preparing calibration input")
    with torch.no_grad():
        inps, outs, attention_mask, position_ids = prepare_calibration_input(model, dataloader)
    print("calibration input prepared")

    layers = get_layers(model)
    
    for i in range(len(layers)):
        layer = layers[i]
        subset = find_layers(layer, layers=(Linear,))

        # Handle model parallel device map (optional)
        if hasattr(model, "hf_device_map") and f"model.layers.{i}" in model.hf_device_map:
            dev = model.hf_device_map[f"model.layers.{i}"]
            inps = inps.to(dev)
            outs = outs.to(dev)
            attention_mask = attention_mask.to(dev) if attention_mask is not None else None
            position_ids = position_ids.to(dev) if position_ids is not None else None
        else:
            dev = inps.device

        # Wrap linears to collect activation stats
        wrapped_layers = {name: WrappedGPT(mod.base_layer, layer_id=i, layer_name=name) for name, mod in subset.items()}

        def add_batch(name):
            def _hook(mod, inp, out):
                # inp is a tuple; inp[0] is the tensor input to this Linear
                wrapped_layers[name].add_batch(inp[0].detach(), out.detach())
            return _hook

        handles = []
        for name, mod in subset.items():
            handles.append(mod.register_forward_hook(add_batch(name)))

        # Run layer forward nsamples times to populate wrapped_layers[name].scaler_row
        # IMPORTANT: Use Qwen3-correct layer forward (provides position_embeddings)
        for j in range(nsamples):
            hs = inps[j].unsqueeze(0)  # [1, seqlen, hidden]
            # vector = layer_forward(
            #     model=model,
            #     layer=layer,
            #     hidden_states=hs,
            #     attention_mask=attention_mask,
            #     position_ids=position_ids,
            # ).squeeze(0)
            # print("vector shape:", vector.shape)
            outs[j] = layer_forward(
                model=model,
                layer=layer,
                hidden_states=hs,
                attention_mask=attention_mask,
                position_ids=position_ids,
            ).squeeze(0)

        for h in handles:
            h.remove()

        #please edit the following part to compute correct importance score and add it to score, set mask based on the importance score
        for name in subset:
            # Access the relevant tensors for each module.
            print(f"Pruning layer {i}, param {name}")
            weight       = subset[name].base_layer.weight
            lora_A       = subset[name].lora_A.default.weight
            lora_B       = subset[name].lora_B.default.weight
            if not hasattr(subset[name], "score") or subset[name].score is None:
                subset[name].score = torch.zeros_like(weight, device=torch.device("cpu"))
            score_tensor = subset[name].score
            # X_hat is computed from the activation statistics (assumed to be feature norm)
            X_hat = torch.sqrt(wrapped_layers[name].scaler_row.reshape((1, -1))).to(mod.weight.device)
            # Save the original devices for score and mask
            orig_score_device = score_tensor.device

            # Move score and mask to the current computation device (assume weight's device)
            device = weight.device
            if score_tensor.device != device:
                score_tensor = score_tensor.to(device)
            # print(f"Module: {name}")
            # print(f"  weight shape:           {weight.shape}")
            # print(f"  lora_A shape:           {lora_A.shape}")
            # print(f"  lora_B shape:           {lora_B.shape}")
            # print(f"  X_hat shape:            {X_hat.shape}")

            # Determine the complementary effective weight.
            if name == "self_attn.q_proj":
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                effective_weight_Q = effective_weight.view(num_q_heads, head_dim, -1)
                abs_Q = torch.abs(effective_weight_Q)
                comp_layer = subset["self_attn.k_proj"]
                comp_weight = comp_layer.base_layer.weight + torch.matmul(
                    comp_layer.lora_B.default.weight,
                    comp_layer.lora_A.default.weight,
                )
                effective_weight_K = comp_weight.view(num_kv_groups, head_dim, -1)
                kv_group_norms = torch.norm(effective_weight_K, p=2, dim=1)
                group_size = num_q_heads // num_kv_groups
                kv_group_idx = torch.arange(num_q_heads, device=abs_Q.device) // group_size  # [num_q_heads]
                kv_norms_expanded = kv_group_norms[kv_group_idx]
                X_hat = X_hat.view(1, 1, -1).to(abs_Q.device)
                importance = torch.mul(abs_Q, kv_norms_expanded.unsqueeze(1))
                importance = torch.mul(importance, X_hat)
                importance = importance.view(-1, importance.shape[-1])

            if name == "self_attn.k_proj":
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                k_weight_grouped = effective_weight.view(num_kv_groups, head_dim, -1)
                k_weight_abs = torch.abs(k_weight_grouped)
                comp_layer = subset["self_attn.q_proj"]
                q_weight = comp_layer.base_layer.weight + torch.matmul(
                    comp_layer.lora_B.default.weight,
                    comp_layer.lora_A.default.weight,
                )
                group_size = num_q_heads // num_kv_groups
                q_weight_grouped = q_weight.view(num_kv_groups, group_size, head_dim, -1)
                q_head_norm = torch.norm(q_weight_grouped, p=2, dim=2)
                q_group_norm = torch.sqrt(torch.mean(torch.mul(q_head_norm, q_head_norm), dim=1, keepdim=True))
                X_hat = X_hat.view(1, 1, -1).to(k_weight_abs.device)
                importance_grouped = torch.mul(k_weight_abs, q_group_norm)
                importance_grouped = torch.mul(importance_grouped, X_hat)
                importance = importance_grouped.view(-1, importance_grouped.shape[-1])

            elif name == "self_attn.v_proj":
                # Compute effective v_proj weight: shape [512, 2048] → 8 heads × 64 rows
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                out_features, in_features = effective_weight.shape
                g = num_q_heads // num_kv_groups
                Wv_h = effective_weight.view(num_kv_groups, head_dim, in_features)  # [h_kv, d_h, in_features]
                comp_layer = subset["self_attn.o_proj"]
                comp_weight = comp_layer.base_layer.weight + torch.matmul(
                                    comp_layer.lora_B.default.weight,
                                    comp_layer.lora_A.default.weight)
                
                Wo_blocks = comp_weight.view(comp_weight.shape[0], num_q_heads, head_dim)
                Wo_grouped = Wo_blocks.view(comp_weight.shape[0], num_kv_groups, g, head_dim)
                tmp = torch.norm(Wo_grouped, p=2, dim=0)          # norm over out_dim -> [h_kv, g, d_h]
                comp_kv = torch.sqrt(tmp.pow(2).mean(dim=1))       # RMS over g -> [h_kv, d_h]
                v_proj_head_rows = effective_weight.shape[0] // num_kv_groups
                comp_head_rows = comp_weight.shape[0] // num_kv_groups
                o_proj_grouped = comp_weight.view(num_kv_groups, comp_head_rows, comp_weight.shape[-1])
                comp_norm = torch.norm(o_proj_grouped, p=2, dim=1, keepdim=True)  # shape: (8, 1, 2048)
                importance_grouped = torch.abs(Wv_h) * comp_kv.unsqueeze(-1) * X_hat  # shape: (8, 64, 2048)
                importance = importance_grouped.view(out_features, in_features)

            elif name == "self_attn.o_proj":
                # Compute o_proj effective weight: (2048, 2048)
                effective_weight = weight + torch.matmul(lora_B, lora_A)

                norm_gate = get_comp_norm(comp = subset["mlp.gate_proj"], dim=0)
                norm_up   = get_comp_norm(comp = subset["mlp.up_proj"], dim=0)
                norm = (norm_gate + norm_up)/2
                norm = norm.reshape(-1)
                print(f"effective_weight shape:    {effective_weight.shape}")
                print(f"X_hat shape:               {X_hat.shape}")
                print(f"  norm shape:                 {norm.shape}")
                print(f"  norm_gate shape:            {norm_gate.shape}")   
                print(f"  norm_up shape:              {norm_up.shape}")
                # Compute importance as per:
                # importance_{ij} = |(W1+B1A1)_{ij}| * ||X_{:,i}||_2 * (norm_gate + norm_up)_j
                importance = torch.abs(effective_weight) * X_hat * norm
                # importance = torch.abs(effective_weight) * X_hat 
            elif name == "mlp.gate_proj":
                # Compute effective gate_proj weight: shape [8192, 2048]
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                
                comp_norm = get_comp_norm(comp = subset["mlp.down_proj"], dim=1)
                # print(f"  compt_norm shape:            {comp_norm.shape}")
                # Multiply elementwise:
                # - effective_weight: shape (8192, 2048) (absolute value applied elementwise)
                # - comp_norm: shape (8192, 1), broadcast along the columns
                # - X_hat: shape (1, 2048), broadcast along the rows.
                importance = torch.abs(effective_weight) * comp_norm * X_hat  # shape: (8192, 2048)
            elif name == "mlp.up_proj":
                # effective_weight: shape [8192, 2048]
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                
                # Get the complementary weight from mlp.down_proj.
                # mlp.down_proj has shape [2048, 8192]: its columns (8192) correspond to the expanded neurons.
                comp_norm = get_comp_norm(comp = subset["mlp.down_proj"], dim=1)
                # print(f"  compt_norm shape:            {comp_norm.shape}")
                
                importance = torch.abs(effective_weight) * comp_norm * X_hat
            elif name == "mlp.down_proj":
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                importance = torch.abs(effective_weight) * X_hat
            else:
                print("This weight is not being recognized " + name)
                comp_weight = None

            # print(f"  importance shape: {importance.shape}")
            
            if torch.count_nonzero(score_tensor) > 0:
                # Moving average: new_score = mask_lr * current + (1 - mask_lr) * previous
                print("we use moving average")
                importance = mask_lr * importance + (1 - mask_lr) * score_tensor
            # Update the stored score.
            score_tensor.copy_(importance)

            # merge the lora adpater, apply the mask, and reinitialized the lora adapter
            merge_lora_into_base(subset[name], adapter="default")
            reset_lora(subset[name], adapter="default")
            
            # Rowwise Selection
            W_mask = torch.zeros_like(importance, dtype=torch.bool)
            num_to_prune = int(importance.shape[1] * prune_ratio)
            sort_res = torch.sort(importance, dim=-1, stable=True)
            indices = sort_res[1][:, :num_to_prune]
            # Scatter True into W_mask for the lowest 'num_to_prune' positions in each row
            rows = torch.arange(importance.shape[0]).unsqueeze(-1).to(importance.device)
            W_mask[rows, indices] = True
            weight.masked_fill_(W_mask, 0.0)

            if score_tensor.device != orig_score_device:
                score_tensor = score_tensor.to(orig_score_device)

        for j in range(min(128, inps.shape[0])):
            with torch.no_grad():
                pos_emb = model.base_model.model.model.rotary_emb(x = inps[j].unsqueeze(0),position_ids = position_ids)
                outs[j] = layer(
                    inps[j].unsqueeze(0),
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    position_embeddings=pos_emb
                )[0]
        inps, outs = outs, inps

    # Restore original cache setting
    model.config.use_cache = use_cache
    torch.cuda.empty_cache()

def get_comp_norm(comp, dim):
    comp_weight = comp.base_layer.weight + torch.matmul(comp.lora_B.default.weight,
                                                        comp.lora_A.default.weight)
    # comp_weight has shape [8192, 2048]. Compute L2 norm over dim=0, yielding (1,2048),
    # then transpose to get (2048,1)
    # return torch.norm(comp_weight, p=2, dim=0, keepdim=True).transpose(0, 1).flip(dims=[0])
    # return torch.norm(comp_weight, p=2, dim=0, keepdim=True).transpose(0, 1)
    return torch.norm(comp_weight, p=2, dim=dim).reshape(1, -1)


In [3]:
from pruning.utils import check_sparsity

foresight_prune(peft_model,
                prune_ratio=0.2,
                mask_lr=0.5,
                nsamples=128,
                seed=42,
                tokenizer=tokenizer)
check_sparsity(peft_model)


loading data


Token indices sequence length is longer than the specified maximum sequence length for this model (528225 > 131072). Running this sequence through the model will result in indexing errors


dataset loading complete
preparing calibration input
calibration input prepared
Pruning layer 0, param self_attn.q_proj
This weight is not being recognized self_attn.q_proj
Pruning layer 0, param self_attn.k_proj
Pruning layer 0, param self_attn.v_proj
Pruning layer 0, param self_attn.o_proj
effective_weight shape:    torch.Size([1024, 2048])
X_hat shape:               torch.Size([1, 2048])
  norm shape:                 torch.Size([1024])
  norm_gate shape:            torch.Size([1, 1024])
  norm_up shape:              torch.Size([1, 1024])


RuntimeError: The size of tensor a (2048) must match the size of tensor b (1024) at non-singleton dimension 1